<a href="https://colab.research.google.com/github/ekrombouts/GenCareAI/blob/work_in_progress/scripts/work_in_progress/400_CarePlan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summarisation: SAMPC

**Author:** Eva Rombouts  
**Date:** 2024-09-21  

### Description
This script performs data cleaning and summarisation of nursing reports, using a Pydantic model (`SAMPC`) to extract and structure important fields like physical complaints, ADL (Activities of Daily Living), mobility, incontinence, social and family details, and psychological issues. The summaries are generated by leveraging an OpenAI model for natural language understanding and processing. The final output is a DataFrame containing client IDs, the month of the report, and the summarised data in SAMPC format.


In [1]:
!pip install -q datasets langchain langchain_community langchain_openai
from google.colab import drive, userdata

import pandas as pd
from datasets import load_dataset
from typing import List

from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.callbacks import get_openai_callback
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 488.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.1/374.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194

In [2]:
# Mount Google Drive to access the Hugging Face token and OpenAI API key
drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_TOKEN')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Set model parameters
model = "gpt-4o-mini-2024-07-18"
# model="gpt-4o-2024-08-06")
temperature = 1.0

Mounted at /content/drive


In [3]:
# Load dataset from Hugging Face
dataset = load_dataset("ekrombouts/Galaxy_records", token=HF_TOKEN)
df_records = dataset['train'].to_pandas()

README.md:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.33M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29668 [00:00<?, ? examples/s]

In [5]:
# Floor datetime to the first day of the month
df_records['month'] = df_records['datetime'].dt.to_period('M').dt.to_timestamp()

# Group records by 'ct_id' and 'month', concatenating all notes into one string
df = (
    df_records
    .groupby(['ct_id', 'month'])
    .agg({'note': lambda x: '\n'.join(x)})
    .reset_index()
    .rename(columns={'note': 'notes'})
)

In [6]:
# Define the SAMPC model using Pydantic to structure the summarised data
class SAMPC(BaseModel):
    somatiek: List[str] = Field(description="lichamelijke klachten")
    adl: str = Field(description="beschrijf welke hulp de cliënt nodig heeft bij wassen en kleden")
    mobiliteit: str = Field(description="beschrijf de mobiliteit (bv rolstoelafhankelijk, gebruik rollator, valgevaar)")
    continentie: str = Field(description="continentie")
    maatschappelijk: str = Field(description="beschrijf bijzonderheden familie en dagbesteding")
    psychisch: List[str] = Field(description="beschrijf cognitie en probleemgedrag")

In [7]:
# Initialize OpenAI Chat model
model = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=temperature, model=model)

In [8]:
# Set up a parser to handle the output and inject instructions into the prompt template
pyd_parser = PydanticOutputParser(pydantic_object=SAMPC)

In [9]:
prompt_template = PromptTemplate(
    template = """
Vat de onderstaande rapportages kort samen op de volgende manier:
- Somatiek
- Wassen en aankleden
- Mobiliteit
- Continentie
- Maatschappelijk
- Bijzonderheden familie en dagbesteding
- Psychisch

---
RAPPORTAGES:
{rapportages}
---

{format_instructions}
""",
    input_variables=["rapportages"],
    partial_variables={"format_instructions": pyd_parser.get_format_instructions()},
)

# Example prompt formatting
prompt = prompt_template.format(rapportages="hier komen de rapportages")
print(prompt)


Vat de onderstaande rapportages kort samen op de volgende manier:
- Somatiek
- Wassen en aankleden
- Mobiliteit
- Continentie
- Maatschappelijk
- Bijzonderheden familie en dagbesteding
- Psychisch

---
RAPPORTAGES:
hier komen de rapportages
---

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"somatiek": {"description": "lichamelijke klachten", "items": {"type": "string"}, "title": "Somatiek", "type": "array"}, "adl": {"description": "beschrijf welke hulp de cliënt nodig heeft bij wassen en kleden", "title": "Adl", "type": "string"}, "mobiliteit": {"description": "beschrijf

In [10]:
# Create a chain: prompt -> model -> output parser
chain = prompt_template | model | pyd_parser

In [11]:
# Function to generate the SAMPC summary
def generate_sampc_summary(notes: str) -> SAMPC:
    result = chain.invoke({"rapportages": notes})
    return result

In [12]:
# Create an empty list to store SAMPC results
sampc_results = []

with get_openai_callback() as cb:
  # Loop through the 'notes' column and generate SAMPC summaries, store them in the list
  for notes in df['notes']:
      sampc_summary = generate_sampc_summary(notes)
      sampc_results.append(sampc_summary)
  print(cb)


Tokens Used: 1229169
	Prompt Tokens: 1124163
	Completion Tokens: 105006
Successful Requests: 414
Total Cost (USD): $0.23162804999999992


In [14]:
# Convert SAMPC results into a DataFrame
df_sampc = pd.DataFrame([s.dict() for s in sampc_results])
df_sampc['ct_id'] = df['ct_id']
df_sampc['month'] = df['month']
df_sampc['notes'] = df['notes']

In [17]:
df_sampc.to_csv('df_sampc.csv', index=False)

In [15]:
print(df_sampc.head())

                                            somatiek  \
0  [Geen tekenen van pijn of ongemak., Bloeddruk ...   
1  [Meneer heeft last van hoofdpijn en vermoeidhe...   
2  [Meneer heeft last van duizeligheid bij het op...   
3  [Moeite met opstaan en lopen., Vermoeidheid ti...   
4  [Bloeddruk- en bloedsuikerwaarden gemeten en i...   

                                                 adl  \
0  Meneer had wat hulp nodig bij het wassen en aa...   
1  Cliënt heeft hulp nodig bij aankleden en eten....   
2  Cliënt heeft hulp nodig bij het wassen, aankle...   
3  Cliënt had regelmatig hulp nodig bij het wasse...   
4  Cliënt had hulp nodig bij het wassen, aanklede...   

                                          mobiliteit  \
0  Meneer gebruikt een rollator voor zijn mobilit...   
1  Cliënt gebruikt een rollator. Er is extra onde...   
2  Cliënt maakt gebruik van een rollator, vertoon...   
3  Cliënt maakt gebruik van een rollator, er is e...   
4  Cliënt gebruikt een rollator voor ondersteu

In [16]:
df_sampc.to_csv('df_sampc.csv', index=False)

In [18]:
import os
os.getcwd()

'/content'

In [19]:
df_sampc.to_csv('/content/drive/My Drive/df_sampc.csv', index=False)